In [ ]:
#  HoroConsultant - Production Cloud Fine-Tuning Pipeline
import os
import sys
import subprocess

# Suppress PyDev / frozen modules debugger warnings & force UTF-8 encoding
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['PYTHONIOENCODING'] = 'utf-8:surrogateescape'
os.environ['PYTHONUTF8'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['TQDM_DISABLE'] = '1'
if hasattr(sys.stdout, 'reconfigure'):
    try: sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    except Exception: pass
if hasattr(sys.stderr, 'reconfigure'):
    try: sys.stderr.reconfigure(encoding='utf-8', errors='replace')
    except Exception: pass

# 0. Set CUDA stability env vars FIRST (before any torch/bnb imports)
# Tesla T4 = sm_75 (Compute Capability 7.5) -- bfloat16 requires sm_80+
os.environ.setdefault('TORCH_CUDA_ARCH_LIST', '7.5')     # Target T4 arch
os.environ.pop('BNB_CUDA_VERSION', None)                   # Let BNB auto-detect native CUDA library
os.environ.setdefault('CUDA_MODULE_LOADING', 'LAZY')      # Lazy loading prevents JIT errors at import
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('[OK] CUDA stability environment variables set (T4/sm_75 compatible)')

# 1. Load Secrets safely from Kaggle Secrets (individual try-except per key)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    for secret_key in ['HF_TOKEN', 'APP_SUPABASE_URL', 'APP_SUPABASE_KEY', 'GH_TOKEN']:
        try:
            val = user_secrets.get_secret(secret_key)
            if val:
                os.environ[secret_key] = val
                print(f'[OK] Kaggle Secret loaded: {secret_key}')
        except Exception as e:
            print(f'[INFO] Kaggle Secret note ({secret_key}): {e}')
except Exception as e:
    print(f'[INFO] Kaggle Secrets Client not available: {e}')

# 2. Safe Git Clone / Pull with pure Python subprocess
target_dir = '/kaggle/working/HoroConsultant'
if not os.path.exists(target_dir):
    print('[MODEL] Cloning HoroConsultant repository...')
    subprocess.run(['git', 'clone', 'https://github.com/pphothidaen/HoroConsultant.git', target_dir], check=True)
else:
    print('[SYNC] Resetting and pulling latest updates...')
    subprocess.run(['git', '-C', target_dir, 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', target_dir, 'reset', '--hard', 'origin/main'], check=True)

os.chdir(target_dir)
if target_dir not in sys.path:
    sys.path.insert(0, target_dir)

# 3. Install Fine-Tuning Dependencies preserving Kaggle's pre-installed CUDA PyTorch
print('[CHECK] Checking pre-installed PyTorch & CUDA status...')
import torch
print(f'[CUDA] Kaggle PyTorch version: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    dev_name = torch.cuda.get_device_name(0)
    target_sm = f'sm_{cap[0]}{cap[1]}'
    print(f'[CUDA] Detected GPU: {dev_name} ({target_sm})')
    if cap == (6, 0):
        print('[WARNING] Kaggle allocated Tesla P100 (sm_60). Installing PyTorch 2.2.0+cu121 compatibility wheel with native sm_60 Pascal support...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', '--no-deps', 'torch==2.2.0+cu121', 'torchvision==0.17.0+cu121', '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
    else:
        print(f'[OK] {dev_name} ({target_sm}) fully supported by native PyTorch.')
print('[MODEL] Removing incompatible torchao/torchvision & installing fine-tuning packages...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao', 'torchvision'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--progress-bar', 'off', '--prefer-binary', '--no-deps', 'transformers==4.44.2', 'tokenizers==0.19.1', 'peft==0.12.0', 'trl==0.11.0', 'accelerate==0.33.0', 'bitsandbytes==0.43.3', 'datasets==2.18.0', 'huggingface_hub==0.25.1', 'pyarrow_hotfix'], check=True)
try:
    import bitsandbytes
    bnb_ver = getattr(bitsandbytes, '__version__', 'unknown')
except Exception as bnb_e:
    bnb_ver = f'bypassed ({bnb_e})'
import transformers, peft, trl, datasets, accelerate
print(f'[OK] Fail-Fast Import Verified: transformers={transformers.__version__}, peft={peft.__version__}, trl={trl.__version__}, accelerate={accelerate.__version__}, bitsandbytes={bnb_ver}')

# 4. Run Cloud Training Orchestrator with execution logging
# Pass the full environment (incl. CUDA stability vars) to subprocess
print('\ud83d\ude80 Launching Cloud Training Orchestrator...')
log_path = '/kaggle/working/train_execution.log'
train_env = os.environ.copy()
train_env['PYTHONIOENCODING'] = 'utf-8:surrogateescape'
train_env['PYTHONUTF8'] = '1'
proc = subprocess.Popen([sys.executable, 'scripts/cloud_train_orchestrator.py', '--platform', 'KAGGLE_T4X2', '--base-model', 'Qwen/Qwen2.5-7B-Instruct', '--epochs', '3'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace', env=train_env)
with open(log_path, 'w', encoding='utf-8', errors='replace') as log_f:
    for line in iter(proc.stdout.readline, ''):
        safe_line = line.encode('utf-8', errors='replace').decode('utf-8', errors='replace')
        sys.stdout.write(safe_line)
        log_f.write(safe_line)
proc.wait()
if proc.returncode != 0:
    log_tail = ''
    if os.path.exists(log_path):
        try:
            with open(log_path, 'r', encoding='utf-8') as f:
                log_tail = ''.join(f.readlines()[-30:])
        except Exception:
            pass
    raise RuntimeError(f'\u274c Training orchestrator failed (exit code {proc.returncode}).\n--- Tail of train_execution.log ---\n{log_tail}')
print('\ud83c\udf89 Training pipeline completed successfully!')
